# Day 1-03｜球場 Keypoint 標註規格
> Python 籃球運動資料分析課程  
> 建立穩定的點位名稱與順序，避免後續 Homography 因標註順序錯誤而失效。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 認識 keypoint dataset 的點位命名與順序要求。
- 比較學生標註與參考答案的 pixel 誤差。
- 判斷標註品質是否足以進入模型訓練或投影流程。

## 完成產出
- 一份簡易標註誤差表與通過/修正建議。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 確認 keypoint 名稱與順序。
2. 輸入或讀取學生標註結果。
3. 計算誤差並檢查是否需要重標。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


## 建議標註任務

請老師準備 10 張固定圖片，學生在 Roboflow 裡標球場參考點。  
這堂只要求：

1. 知道每個 keypoint 的名稱。
2. 依照固定順序點選。
3. 匯出 Dataset Version。
4. 下載或用 Roboflow API 讀到 Colab。

課堂可使用籃球場圖片、紙張四角、桌面角點或其他平面物體作為標註材料。

In [ ]:
# 課堂示範用：先列出一組簡化 keypoint 名稱。
keypoint_names = [
    "paint_left_top",
    "paint_right_top",
    "paint_right_bottom",
    "paint_left_bottom",
]

for i, name in enumerate(keypoint_names):
    print(i, name)

In [ ]:
# 假設學生標的點位如下；老師答案如下。
# 實際課堂可以把 Roboflow 匯出的 annotation 讀進來後換掉這兩個變數。
import numpy as np
import pandas as pd

student_points = np.array(
    [[818, 480], [1215, 455], [1412, 660], [675, 625]], dtype=float
)
answer_points = np.array(
    [[818, 480], [1216, 453], [1410, 657], [672, 623]], dtype=float
)
errors = np.linalg.norm(student_points - answer_points, axis=1)

df = pd.DataFrame(
    {
        "keypoint": keypoint_names,
        "student_x": student_points[:, 0],
        "student_y": student_points[:, 1],
        "answer_x": answer_points[:, 0],
        "answer_y": answer_points[:, 1],
        "error_px": errors,
    }
)
df

In [ ]:
print("mean error:", float(errors.mean()))
print("max error :", float(errors.max()))

# 門檻只是課堂練習用，不是正式研究指標。
threshold = 12
if errors.mean() < threshold:
    print("通過：平均誤差低於門檻")
else:
    print("請檢查點位順序或是否點到錯誤位置")

## 延伸：Keypoint Model 訓練流程

以下區塊保留模型訓練的程式骨架，預設不執行。課堂重點為資料集格式、點位順序與訓練流程的銜接方式；若需要示範推論，可使用教師提供的模型權重。

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    # !pip install -q ultralytics roboflow
    # from roboflow import Roboflow
    # rf = Roboflow(api_key='YOUR_API_KEY')
    # project = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT')
    # dataset = project.version(1).download('yolov8')
    #
    # from ultralytics import YOLO
    # model = YOLO('yolo11n-pose.pt')
    # model.train(data=f'{dataset.location}/data.yaml', epochs=20, imgsz=640)
    pass
else:
    print("目前是教學模式：不訓練，只保留訓練骨架。")